# Text2ToBI — Run Summary

Scans a `runs/` directory tree, parses every `hparams*.json` it finds,
and exports a formatted `.xlsx` summary table.

**Usage:** Edit `RUNS_ROOT` and `OUTPUT_PATH` in Cell 1, then run all cells.
Call `build_summary(sort_by=..., ascending=False)` to re-sort without re-scanning.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Configuration                                      ║
# ╚══════════════════════════════════════════════════════════════╝

MODELS_ROOT         = "/content/drive/MyDrive/Capstone/project/models"
MODEL_REGISTRY_PATH = f"{MODELS_ROOT}/model_registry.json"
OUTPUT_PATH         = f"{MODELS_ROOT}/runs_summary.xlsx"

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

print("✓ Config loaded.")
print(f"  Models root : {MODELS_ROOT}")
print(f"  Registry    : {MODEL_REGISTRY_PATH}")
print(f"  Output      : {OUTPUT_PATH}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Imports                                            ║
# ╚══════════════════════════════════════════════════════════════╗
import json, math, os
from pathlib import Path

from openpyxl import Workbook
from openpyxl.styles import (
    Alignment, Font, PatternFill, Border, Side
)
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.utils import get_column_letter

print("✓ Imports OK.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Registry-driven scanner                            ║
# ║                                                              ║
# ║  Reads model_registry.json for run specs (no more folder-    ║
# ║  name guessing), then opens each run's hparams.json for      ║
# ║  metrics. One row per registry entry.                        ║
# ╚══════════════════════════════════════════════════════════════╗

_FULL_SET = {"libri", "ps", "sbc"}

def _nan_to_dash(v):
    """Return "—" for NaN/None, else round float to 4dp."""
    if v is None:
        return "—"
    if isinstance(v, float):
        if math.isnan(v):
            return "—"
        return round(v, 4)
    return v


def _best_epoch(history_list):
    """1-based epoch index of the max value in a list, or "—"."""
    if not history_list:
        return "—"
    valid = [(i, v) for i, v in enumerate(history_list)
             if isinstance(v, float) and not math.isnan(v)]
    if not valid:
        return "—"
    best_i = max(valid, key=lambda x: x[1])[0]
    return best_i + 1


def _pos_label(train_on_text, train_on_pos):
    if train_on_pos and not train_on_text:
        return "pos_only"
    elif train_on_pos and train_on_text:
        return "pos"
    else:
        return "none"


def load_registry(registry_path):
    if not os.path.exists(registry_path):
        raise FileNotFoundError(f"Registry not found: {registry_path}")
    with open(registry_path) as f:
        return json.load(f)


def scan_runs(models_root, registry_path):
    """
    Read model_registry.json, then open each run's hparams.json to pull
    metrics. Returns one row dict per registry entry.

    Raises FileNotFoundError if the registry is missing or empty.
    Skips (with a warning) any registry entry whose hparams.json is missing
    or fails to parse — does not abort the whole scan for one bad run.
    """
    registry = load_registry(registry_path)
    if not registry:
        raise FileNotFoundError(f"Registry at {registry_path} is empty.")

    rows = []
    skipped = []

    for model_num in sorted(registry, key=int):
        entry = registry[model_num]
        run_dir = os.path.join(models_root, entry["path"])
        hparams_path = os.path.join(run_dir, "hparams.json")

        if not os.path.exists(hparams_path):
            skipped.append((model_num, entry["name"], "hparams.json not found"))
            continue

        try:
            with open(hparams_path, encoding="utf-8") as fh:
                h = json.load(fh)
        except json.JSONDecodeError as e:
            skipped.append((model_num, entry["name"], f"JSON parse error: {e}"))
            continue

        # ── Run specs (from registry — no guessing) ────────────────────────
        datasets   = entry["datasets"]
        full_part  = "full" if set(datasets) == _FULL_SET else "partial"
        corpus_str = "+".join(sorted(datasets))
        if entry["loss"] == "weighted":
            _blw = entry.get("boundary_loss_weight")
            loss = f"wl{int(_blw) if _blw == int(_blw) else _blw}" if _blw is not None else "wl?"
        else:
            loss = "stl"
        pos        = _pos_label(entry["train_on_text"], entry["train_on_pos"])
        punct      = "N" if entry.get("strip_punctuation", True) else "Y"
        lr         = entry["learning_rate"]
        warmup     = entry["warmup_steps"]  # default 0, not "—"

        # ── Metrics (from hparams.json) ─────────────────────────────────────
        res  = h.get("results", {})
        test = res.get("test", {})
        b    = test.get("boundary",   {})
        i_   = test.get("intonation", {})
        vh   = res.get("val_history",   {})
        th   = res.get("train_history", {})
        bu   = res.get("bu_eval") or {}
        ic   = res.get("intonation_checkpoint", {})

        best_epoch_b = h.get("training", {}).get("best_epoch", _best_epoch(vh.get("boundary_f1", [])))

        row = {
            "MODEL_#":     int(model_num),
            "FULL_PART":   full_part,
            "LOSS":        loss,
            "PUNCT":       punct,
            "POS":         pos,
            "B_F1":        _nan_to_dash(b.get("f1")),
            "I_F1":        _nan_to_dash(i_.get("f1")),
            "X_F1_BU":     _nan_to_dash(bu.get("f1")),
            "CORPUS":      corpus_str,
            "LR":          lr,
            "WARMUP":      warmup,
            "SEED":        h.get("data", {}).get("split_seed", "—"),
            "DUAL_CKPT":   bool(ic.get("enabled", False)),
            "EPOCH_B":     best_epoch_b,
            "EPOCH_I":     _best_epoch(vh.get("intonation_f1", [])),
            "EPOCH_X":     _best_epoch(vh.get("break_idx_f1",  [])),
            "B_PREC":      _nan_to_dash(b.get("precision")),
            "B_REC":       _nan_to_dash(b.get("recall")),
            "I_PREC":      _nan_to_dash(i_.get("precision")),
            "I_REC":       _nan_to_dash(i_.get("recall")),
            "X_PREC":      _nan_to_dash(bu.get("precision")),
            "X_REC":       _nan_to_dash(bu.get("recall")),
            "VAL_B_F1":    _nan_to_dash(res.get("best_val_boundary_f1")),
            "TRAIN_B_F1":  _nan_to_dash(
                                th.get("boundary_f1", [None])[
                                    (best_epoch_b - 1) if isinstance(best_epoch_b, int) else 0
                                ] if th.get("boundary_f1") else None
                            ),
            "WORDS":       h.get("data", {}).get("n_words_train_total", "—"),
        }
        rows.append(row)

    if skipped:
        print(f"⚠ Skipped {len(skipped)} registry entr{'y' if len(skipped)==1 else 'ies'}:")
        for model_num, name, reason in skipped:
            print(f"    Model #{model_num} ({name}): {reason}")

    if not rows:
        raise FileNotFoundError(
            f"No valid runs found. Registry had {len(registry)} entries, "
            f"all skipped — check {models_root} paths."
        )

    return rows


print("✓ Cell 3: registry-driven scanner defined.")
print("  scan_runs(models_root, registry_path) → list of row dicts")


In [ ]:
# Quick test — scan the real registry and inspect the first couple of rows.
test_rows = scan_runs(MODELS_ROOT, MODEL_REGISTRY_PATH)
print(f"\n✓ Scanned {len(test_rows)} run(s).\n")

for row in test_rows[:2]:
    print(f"--- Model #{row['MODEL_#']} ---")
    for k, v in row.items():
        print(f"  {k:12s}: {v}")
    print()

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — xlsx writer                                        ║
# ╚══════════════════════════════════════════════════════════════╗

# Column definitions: (key, header label, width)
# Order: run specs → headline results → secondary specs → extended detail
COLUMNS = [
    ("MODEL_#",     "MODEL_#",     9),
    ("FULL_PART",   "FULL/PART",  10),
    ("LOSS",        "LOSS",        7),
    ("PUNCT",       "PUNCT",       6),
    ("POS",         "POS",        10),
    ("B_F1",        "B_F1 ▼",     10),   # primary sort column
    ("I_F1",        "I_F1",        9),
    ("X_F1_BU",     "X_F1 (BU)",  11),
    ("CORPUS",      "CORPUS",     16),
    ("LR",          "LR",         10),
    ("WARMUP",      "WARMUP",      8),
    ("SEED",        "SEED",        7),
    ("DUAL_CKPT",   "DUAL_CKPT",  10),
    ("EPOCH_B",     "EPOCH_B",     8),
    ("EPOCH_I",     "EPOCH_I",     8),
    ("EPOCH_X",     "EPOCH_X",     8),
    ("B_PREC",      "B_PREC",      9),
    ("B_REC",       "B_REC",       9),
    ("I_PREC",      "I_PREC",      9),
    ("I_REC",       "I_REC",       9),
    ("X_PREC",      "X_PREC",      9),
    ("X_REC",       "X_REC",       9),
    ("VAL_B_F1",    "VAL_B_F1",   11),
    ("TRAIN_B_F1",  "TRAIN_B_F1", 12),
    ("WORDS",       "WORDS",      12),
]

_HEADER_FILL  = PatternFill("solid", start_color="1F4E79")   # dark blue
_HEADER_FONT  = Font(bold=True, color="FFFFFF", name="Arial", size=10)
_DATA_FONT    = Font(name="Arial", size=10)
_ALT_FILL     = PatternFill("solid", start_color="EBF3FB")   # light blue
_BORDER_SIDE  = Side(style="thin", color="CCCCCC")
_CELL_BORDER  = Border(
    left=_BORDER_SIDE, right=_BORDER_SIDE,
    top=_BORDER_SIDE,  bottom=_BORDER_SIDE
)
_CENTER       = Alignment(horizontal="center", vertical="center")
_LEFT         = Alignment(horizontal="left",   vertical="center")


def write_xlsx(rows, output_path, sort_by="B_F1", ascending=False):
    """
    Write rows to an xlsx file.

    Parameters
    ----------
    rows       : list of dicts from scan_runs()
    output_path: str  — full path including filename
    sort_by    : str  — column key to sort by (default: B_F1)
    ascending  : bool — sort direction (default: False = best first)
    """
    # ── Sort ──────────────────────────────────────────────────────
    def _sort_key(row):
        v = row.get(sort_by, "—")
        if v == "—" or v is None:
            return (-1 if not ascending else float("inf"))
        if isinstance(v, bool):
            return int(v)
        return v

    sorted_rows = sorted(rows, key=_sort_key, reverse=not ascending)

    # ── Workbook ──────────────────────────────────────────────────
    wb = Workbook()
    ws = wb.active
    ws.title = "Run Summary"
    ws.freeze_panes = "B2"   # freeze row 1 (header) + col A (model #)

    # ── Header row ────────────────────────────────────────────────
    for col_idx, (key, label, width) in enumerate(COLUMNS, start=1):
        cell = ws.cell(row=1, column=col_idx, value=label)
        cell.font      = _HEADER_FONT
        cell.fill      = _HEADER_FILL
        cell.alignment = _CENTER
        cell.border    = _CELL_BORDER
        ws.column_dimensions[get_column_letter(col_idx)].width = width

    ws.row_dimensions[1].height = 20

    # ── Data rows ─────────────────────────────────────────────────
    for row_idx, row in enumerate(sorted_rows, start=2):
        is_alt = (row_idx % 2 == 0)
        for col_idx, (key, label, _) in enumerate(COLUMNS, start=1):
            val = row.get(key, "—")
            if isinstance(val, bool):
                val = "Y" if val else "N"
            c = ws.cell(row=row_idx, column=col_idx, value=val)
            c.font      = _DATA_FONT
            c.border    = _CELL_BORDER
            c.alignment = _LEFT if key in ("CORPUS",) else _CENTER
            if is_alt and val != "—":
                c.fill = _ALT_FILL
            if isinstance(val, float):
                c.number_format = "0.0000"

    # ── Conditional formatting on B_F1 only ─────────────────────────
    b_f1_col = next(
        get_column_letter(i + 1)
        for i, (key, _, __) in enumerate(COLUMNS)
        if key == "B_F1"
    )
    n_rows = len(sorted_rows)
    b_f1_range = f"{b_f1_col}2:{b_f1_col}{n_rows + 1}"

    ws.conditional_formatting.add(
        b_f1_range,
        ColorScaleRule(
            start_type="min",  start_color="F8696B",   # red   — low F1
            mid_type="num",    mid_value=0.75,
                               mid_color="FFEB84",     # yellow — mid
            end_type="max",    end_color="63BE7B",     # green  — high F1
        )
    )

    # ── Save ──────────────────────────────────────────────────────
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
    wb.save(output_path)
    print(f"✓ Saved {n_rows} runs → {output_path}")
    print(f"  Sorted by: {sort_by}  ({'asc' if ascending else 'desc'})")


print("✓ Cell 5: xlsx writer defined.")
print("  write_xlsx(rows, output_path, sort_by=..., ascending=False)")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — build_summary()  ← call this cell to run          ║
# ╚══════════════════════════════════════════════════════════════╗

def build_summary(sort_by="B_F1", ascending=False):
    """
    Scan the model registry, pull metrics from each run's hparams.json,
    and write the summary xlsx.

    Parameters
    ----------
    sort_by   : column key to sort by.
                Valid keys: MODEL_#, FULL_PART, LOSS, PUNCT, POS,
                B_F1, I_F1, X_F1_BU, CORPUS, LR, WARMUP, SEED, DUAL_CKPT,
                EPOCH_B, EPOCH_I, EPOCH_X,
                B_PREC, B_REC, I_PREC, I_REC, X_PREC, X_REC,
                VAL_B_F1, TRAIN_B_F1, WORDS
    ascending : sort direction (False = best first)

    Examples
    --------
    build_summary()                          # default: test boundary F1 desc
    build_summary(sort_by="I_F1")            # sort by intonation F1
    build_summary(sort_by="X_F1_BU")         # sort by gold break index F1
    build_summary(sort_by="MODEL_#", ascending=True)
    """
    print(f"Scanning registry at {MODEL_REGISTRY_PATH} ...")
    rows = scan_runs(MODELS_ROOT, MODEL_REGISTRY_PATH)
    print(f"Found {len(rows)} run(s).")
    write_xlsx(rows, OUTPUT_PATH, sort_by=sort_by, ascending=ascending)

# ── Run immediately ───────────────────────────────────────────────
build_summary()